In [0]:
# ==================================================
# BRONZE — Ingestão Raw
# Pipeline PRF Acidentes de Trânsito
# ==================================================

# Caminhos base — Unity Catalog Volumes
BASE_PATH = "/Volumes/workspace/default/prf_acidentes"
BRONZE_PATH = f"{BASE_PATH}/bronze"

# Cria o Volume via SQL primeiro
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.prf_acidentes")

# Cria subpasta bronze
dbutils.fs.mkdirs(BRONZE_PATH)

print(f"Bronze path: {BRONZE_PATH}")
print("Ambiente pronto!")

In [0]:
# ==================================================
# Leitura do CSV raw e escrita no Delta Lake
# ==================================================

BRONZE_TABLE = "workspace.default.bronze_acidentes"

# Lê o CSV com encoding e separador corretos
df_raw = (spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/prf_acidentes/datatran2025.csv")
)

# Adiciona coluna de partição e timestamp de ingestão
from pyspark.sql.functions import lit, current_timestamp

df_raw = (df_raw
    .withColumn("ano_particao", lit(2025))
    .withColumn("ingested_at", current_timestamp())
)

# Salva como Delta Table
(df_raw.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_particao")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze gravada com sucesso!")
print(f"Total de registros: {df_raw.count()}")

In [0]:
# Validação rápida
spark.sql("SELECT ano_particao, COUNT(*) as total FROM workspace.default.bronze_acidentes GROUP BY ano_particao").show()
spark.sql("DESCRIBE DETAIL workspace.default.bronze_acidentes").select("format","numFiles","sizeInBytes").show()